##  Step 1: Import libraries


In [ ]:
import numpy as np  
import pandas as pd
import nltk
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
import gzip
import joblib


##  Step 2: Load dataset


In [3]:
medicine = pd.read_csv('data_files/medicine.csv')
medicine.head()

,index,Drug_Name,Reason,Description
0,1,A CN Gel(Topical) 20gmA CN Soap 75gm,Acne,Mild to moderate acne (spots)
1,2,A Ret 0.05% Gel 20gmA Ret 0.1% Gel 20gmA Ret 0...,Acne,A RET 0.025% is a prescription medicine that i...
2,3,ACGEL CL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
3,4,ACGEL NANO Gel 15gm,Acne,It is used to treat acne vulgaris in people 12...
4,5,Acleen 1% Lotion 25ml,Acne,treat the most severe form of acne (nodular ac...


## Step 3: Basic data Cleaning

In [4]:
print(medicine.shape)

(9720, 4)


In [5]:
print(medicine.isnull().sum())

index          0
Drug_Name      0
Reason         0
Description    0
dtype: int64


In [6]:
medicine.dropna(inplace=True)

In [ ]:
# Removes missing (NaN) and duplicate rows to clean the dataset.
medicine = medicine[~medicine.duplicated()]

### Step 4: Preprocessing - Convert Description and Reason to tokens


🔍 Why we are doing this?
1. Natural Language Processing (NLP) starts with tokenization
Both Reason and Description columns contain textual data (sentences/phrases). To process these texts effectively for machine learning or similarity analysis, we need to tokenize them — break them into smaller units (words).

2. Prepare for Feature Engineering
In later steps, you combine the Reason and Description into a new column tags to build a Bag of Words (BoW) or vector representation using CountVectorizer.

In [8]:
# The .split() method takes a string and splits it into a list of words (also called tokens) by default using whitespace (" ") as the separator.
# x = "Used for treating skin infection"
# x.split() ➞ ['Used', 'for', 'treating', 'skin', 'infection']


medicine['Reason'] = medicine['Reason'].apply(lambda x: x.split())
medicine['Description'] = medicine['Description'].apply(lambda x: x.split())




### Step 5: Remove whitespaces and merge both columns


In [9]:

# Removes spaces inside words like "head ache" → "headache".
medicine['Description'] = medicine['Description'].apply(lambda x: [i.replace(" ","") for i in x])


# Merges description and reason into a new column tags.
medicine['tags'] = medicine['Description'] + medicine['Reason']


###  Step 6: Create a new dataframe


In [10]:

# Keeps only necessary columns: index, Drug_Name, and tags.
new_df = medicine[['index','Drug_Name','tags']]


# Joins word lists into a single lowercase string like "pain fever cough".
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

C:\Users\gaure\AppData\Local\Temp\ipykernel_23280\1193886593.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())


## Step 7: Stemming


In [12]:
# Reduces words to their base form, e.g., “running”, “runs” → “run”, using PorterStemmer.


ps = PorterStemmer()

def stem(text):
    return " ".join([ps.stem(i) for i in text.split()])

new_df['tags'] = new_df['tags'].apply(stem)



C:\Users\gaure\AppData\Local\Temp\ipykernel_23280\1625736297.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem)


### Step 8: Vectorization using Bag of Words (top 5000 words, remove stopwords)


In [14]:
# Converts tags into numeric vectors using Bag-of-Words.
# Ignores common English stopwords like “is”, “the”, “and”.


cv = CountVectorizer(max_features=5000, stop_words='english')

vectors = cv.fit_transform(new_df['tags']).toarray()

###  Step 9: Cosine similarity matrix


In [17]:
# Computes cosine similarity between medicines. Higher score = more similar tags.
similarity = cosine_similarity(vectors)

###  Step 10: Recommendation Function


🔍 Objective:
To recommend 5 medicines that are most similar to the one entered by the user (e.g., "ACGEL CL NANO Gel 15gm"), based on their reason and description using cosine similarity.

In [18]:
def drug_recommendation(medicine_name):

    # This locates the row index in new_df where the Drug_Name matches the given medicine_name.
    medicine_index = new_df[new_df['Drug_Name'] == medicine_name].index[0]


    #  Get all similarity scores for that medicine

    # similarity is a 2D matrix (from cosine_similarity) where each row/column represents a medicine.
    # similarity[medicine_index] returns a list of similarity scores between the input medicine and every other medicine.
    # distances = [1.0, 0.65, 0.22, 0.89, ..., 0.56]
    distances = similarity[medicine_index]


    #  Rank medicines by similarity
    medicines_list = sorted(list(enumerate(distances)), reverse=True, key = lambda x: x[1])[1:6]

    # enumerate(distances) → returns pairs: [(0, 1.0), (1, 0.65), (2, 0.22), ...]
    # First value = index of medicine in new_df
    # Second value = similarity score

    # sorted(..., reverse=True, key=lambda x: x[1])
    # → Sorts by similarity score in descending order.

    #[1:6]
    # → Skips the first one (which is the medicine itself) and picks the next top 5 most similar.




    # Display top 5 recommended medicine names
    for i in medicines_list:
        print(new_df.iloc[i[0]].Drug_Name)



In [19]:
drug_recommendation("ACGEL CL NANO Gel 15gm")


ACGEL NANO Gel 15gm
Acnehit Gel 15gm
Acnelak Soap 75gm
Acnetor AD 1% Ointment 15gm
Acnetor AD Cream 15Acnetor AD Gel 15gm


### Step 11: Save model and similarity matrix


In [20]:
# Save model

# What is this doing?
# Saves the new_df DataFrame (converted to dictionary form using .to_dict()) to a .pkl file using the pickle module.

#  Why use .to_dict()?
# pickle works best with Python native objects. Converting DataFrame to dict makes it lighter and more portable.

pickle.dump(new_df.to_dict(),open('medicine_dict.pkl','wb'))

with gzip.open('similarity.pkl.gz', 'wb') as f:
    pickle.dump(similarity, f, protocol=pickle.HIGHEST_PROTOCOL)

# Save with joblib (also compressed)
joblib.dump(similarity, 'similarity.joblib', compress=1)


['similarity.joblib']